# Surface Mapper Playground

This notebook tests creating a `SurfaceGridRecord` object with `map_surfacegrid()` using normalized SurfaceGrid test JSON input, then exports a flattened JSON representation including inherited attributes.

### Environment setup and imports

Import standard modules and project models/mappers. Paths are configured relative to the repository root so this notebook can run from the project checkout.

In [1]:
import json
from pathlib import Path

from pydantic import ValidationError

from dsis_model_sdk.models.common import SurfaceGrid

from mappers.surface_helpers import normalize_surfacegrid_payload
from mappers.surfacegrid_ow import map_surfacegrid
from models.metadata import InterpretationProcessingMetadata, SourceContext
from tables import flatten_record


In [2]:
repo_root = Path.cwd()
input_path = repo_root / "../tests/data/surfacegrid_volve_public.json"

### Load normalized SurfaceGrid test JSON

Load test JSON, apply normalization used in tests, and confirm critical keys exist.

In [3]:
raw_payload = json.loads(input_path.read_text(encoding="utf-8"))
normalized_payload = normalize_surfacegrid_payload(raw_payload)

required_keys = sorted(
    name for name, field in SurfaceGrid.model_fields.items() if field.is_required()
)

missing_required = [key for key in required_keys if key not in normalized_payload]
assert not missing_required, f"Missing required keys: {missing_required}"

surfacegrid = SurfaceGrid.model_validate(normalized_payload)
print("Loaded and validated SurfaceGrid")
print(
    f"native_uid={surfacegrid.native_uid}, name={surfacegrid.map_data_set_name}, crs={surfacegrid.crs}"
)

Loaded and validated SurfaceGrid
native_uid=2636, name=ihdTHugin13flt3, crs=ST_ED50_UTM31N_P23031_T1133


### Instantiate SurfaceGridRecord from `map_surfacegrid()`

Create processing metadata and source context, then run conversion. If mapper validation fails, the error is shown clearly.

In [ ]:
from datetime import datetime, timezone

processing_metadata = InterpretationProcessingMetadata(
    create_date=datetime(2025, 6, 1, 12, 0, 0, tzinfo=timezone.utc),
    update_date=datetime(2026, 4, 1, 12, 0, 0, tzinfo=timezone.utc),
    file_available=False,
    file_error_message="Failed to download"
)
source_context = SourceContext(
    database="BG4FROST",
    project="VOLVE_PUBLIC",
    timezone="Europe/Berlin",
)

In [8]:
surface = None
try:
    surface = map_surfacegrid(surfacegrid, source_context, processing_metadata)
    print(f"Created object: {type(surface).__name__}")
    print(surface.model_dump_json(indent=2))
except ValidationError as exc:
    print("map_surfacegrid validation failed:")
    print(exc)
    raise

Created object: SurfaceGridRecord
{
  "id": "BG4FROST:VOLVE_PUBLIC:2636",
  "source": {
    "system": "OpenWorks R5000",
    "database": "BG4FROST",
    "project": "VOLVE_PUBLIC",
    "id": "2636",
    "name": "ihdTHugin13flt3",
    "remark": null,
    "create_user": "IHD",
    "update_user": "IHD",
    "create_date": "2013-11-15T08:20:49",
    "create_date_utc": "2013-11-15T07:20:49Z",
    "update_date": "2013-11-15T08:20:49",
    "update_date_utc": "2013-11-15T07:20:49Z"
  },
  "source_ow": {
    "geo_name": "UNKNOWN",
    "geo_type": "SURFACE",
    "attribute": "DEPTH"
  },
  "source_petrel": null,
  "processing": {
    "create_date": null,
    "update_date": null,
    "file_available": null,
    "file_error_message": null,
    "file_path": null
  },
  "extent": null,
  "crs": "ST_ED50_UTM31N_P23031_T1133",
  "z_domain": "TVDSS",
  "z_unit": "meters",
  "geometry": {
    "ncol": 879,
    "nrow": 629,
    "xori": 429588.0,
    "yori": 6475211.0,
    "xinc": 12.0,
    "yinc": 12.0,
  

### Flatten SurfaceGridRecord object to JSON

Flatten nested dictionaries to dot-path keys for deterministic, compact comparison and downstream inspection.

In [9]:
flat_surface = dict(sorted(flatten_record(surface).items(), key=lambda kv: kv[0]))
print(f"Flattened keys: {len(flat_surface)}")
print(json.dumps(dict(list(flat_surface.items())), indent=2))

Flattened keys: 40
{
  "crs": "ST_ED50_UTM31N_P23031_T1133",
  "extent_points": null,
  "geometry_left_handed": true,
  "geometry_ncol": 879,
  "geometry_nrow": 629,
  "geometry_rotation": 0.0,
  "geometry_xinc": 12.0,
  "geometry_xori": 429588.0,
  "geometry_yinc": 12.0,
  "geometry_yori": 6475211.0,
  "grid_nnan": null,
  "grid_ntotal": null,
  "grid_null_value": null,
  "id": "BG4FROST:VOLVE_PUBLIC:2636",
  "parent_surface_id": null,
  "processing_create_date": null,
  "processing_file_available": null,
  "processing_file_error_message": null,
  "processing_file_path": null,
  "processing_update_date": null,
  "source_create_date": "2013-11-15T08:20:49",
  "source_create_date_utc": "2013-11-15T07:20:49Z",
  "source_create_user": "IHD",
  "source_database": "BG4FROST",
  "source_id": "2636",
  "source_name": "ihdTHugin13flt3",
  "source_ow_attribute": "DEPTH",
  "source_ow_geo_name": "UNKNOWN",
  "source_ow_geo_type": "SURFACE",
  "source_petrel_business_project": null,
  "source_pet

### Notes

- This notebook uses the same normalization logic from `surface_helpers.py` as tests (normalizes `rotation_i`, `rotation_j`, `data_min`, `data_max`).